# step-counter-increment composite — cx14: scalar-loss backward followed by step counter tick

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `backward-on-scalar-loss`, `step-counter-increment`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
import wandb

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "step-counter-increment"
DD_ATOM_IDS = ["backward-on-scalar-loss", "step-counter-increment"]
DD_SUBTOPICS = ["PyTorch: backward()", "Trainer: step counter increment"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Every trainer carries a step counter — the x-axis for logs, the trigger for learning-rate schedules, the gate for `if step % checkpoint_every == 0`. The canonical placement is **after** the optimizer update is committed:

```python
loss = loss_fn(model(x), y)
loss.backward()              # atom A: backward-on-scalar-loss.
optimizer.step()
optimizer.zero_grad()
self.step += 1               # atom B: step-counter-increment (AFTER step).
```

**Atom A — `backward-on-scalar-loss`.** The loss must be 0-D for `.backward()` to work without a `gradient=` argument. Same atom as cx13.

**Atom B — `step-counter-increment`.** The counter measures how many UPDATES have been applied. Incrementing BEFORE `optimizer.step()` would mean step 1 reflects the model state from BEFORE step 1 happened — the off-by-one bug that breaks `step % log_every == 0` gates.

**Why these together.** A batch with no backward is a batch that didn't update the model, so it should NOT tick the step counter. The two atoms have to fire in the same control flow: backward succeeded ⇒ optimizer updated ⇒ counter ticks. If the backward raises, the counter must NOT advance.

### Composite Exercise — scalar-loss backward followed by step counter tick

**Atoms exercised together**: `backward-on-scalar-loss`, `step-counter-increment`

Implement `cx14_train_epoch(model, optimizer, loader, loss_fn, start_step)`. ONE epoch.

Inputs:
- `model`, `optimizer`, `loader`, `loss_fn`: usual.
- `start_step`: int — counter value BEFORE this epoch.

For each `(x, y)` in `loader`:
1. `pred = model(x)`; `loss = loss_fn(pred, y)`.
2. `loss.backward()` (atom A — requires scalar loss).
3. `optimizer.step()`.
4. `optimizer.zero_grad()` (default semantics — don't worry about set_to_none here).
5. `step += 1` (atom B — AFTER the optimizer.step).
6. Append `(step, loss.item())` to a log list AFTER the increment (so the entry's step reflects post-update state).

Return `(final_step, log_list)`.

**Contract**: if `loss.backward()` raises for ANY batch, the counter must NOT advance for that batch. (Easiest way: increment AFTER backward, not before — same canonical order.)

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx14_train_epoch(model, optimizer, loader, loss_fn, start_step):
    """One epoch. Returns (final_step, [(step, loss_float), ...])."""
    raise NotImplementedError

def _test_cx14():
    from torch.utils.data import TensorDataset, DataLoader

    # Case A: 6 batches, start_step=0 → final_step=6, 6 log entries (1..6).
    t.manual_seed(0)
    x = t.randn(24, 3)
    y = t.randn(24, 1)
    loader = DataLoader(TensorDataset(x, y), batch_size=4, shuffle=False)
    assert len(loader) == 6
    model = nn.Linear(3, 1)
    opt = t.optim.SGD(model.parameters(), lr=0.05)
    loss_fn = nn.MSELoss()

    final, log = cx14_train_epoch(model, opt, loader, loss_fn, start_step=0)
    assert final == 6, f'expected final_step=6, got {final}'
    assert len(log) == 6, f'expected 6 log entries, got {len(log)}'
    steps = [s for s, _ in log]
    assert steps == [1, 2, 3, 4, 5, 6], (
        f'log steps must be [1..6] (ticked AFTER each optimizer.step); got {steps}'
    )
    for s, lv in log:
        assert isinstance(lv, float), f'log entry loss should be float, got {type(lv).__name__}'

    # Case B: start_step accumulates across calls — epoch 2 picks up where epoch 1 left off.
    final2, log2 = cx14_train_epoch(model, opt, loader, loss_fn, start_step=final)
    assert final2 == 12, f'after epoch 2, final_step should be 12, got {final2}'
    assert [s for s, _ in log2] == [7, 8, 9, 10, 11, 12]

    # Case C: if a batch's backward raises, counter must NOT advance for that batch.
    # Inject a bad batch by using a loss_fn that returns a vector (RuntimeError on backward).
    vec_loss = nn.MSELoss(reduction='none')
    model3 = nn.Linear(3, 1)
    opt3 = t.optim.SGD(model3.parameters(), lr=0.05)
    raised = False
    try:
        cx14_train_epoch(model3, opt3, loader, vec_loss, start_step=100)
    except RuntimeError:
        raised = True
    assert raised, 'expected RuntimeError on vector-loss backward — code must call .backward() directly'

    # Case D: empty loader → counter unchanged, log empty.
    empty_loader = DataLoader(TensorDataset(t.zeros(0, 3), t.zeros(0, 1)), batch_size=4)
    f3, l3 = cx14_train_epoch(model, opt, empty_loader, loss_fn, start_step=42)
    assert f3 == 42, f'empty loader must not advance counter; got {f3}'
    assert l3 == [], f'empty loader must produce empty log; got {l3}'
    _dd_passed.add('cx14')

_test_cx14()

<details><summary>Show solution — cx14</summary>

```python
def cx14_train_epoch(model, optimizer, loader, loss_fn, start_step):
    step = start_step
    log = []
    for x, y in loader:
        pred = model(x)
        loss = loss_fn(pred, y)
        # Atom A: scalar-loss backward.
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        # Atom B: step counter tick — AFTER optimizer.step.
        step += 1
        log.append((step, loss.item()))
    return step, log
```

Placing the increment AFTER `optimizer.step()` (not before) is what makes Case C automatic — a backward that raises propagates out before the increment line runs, so the counter never advances for the failed batch. If you increment BEFORE backward (some frameworks do this for 'we attempted N batches' semantics), you'd need an explicit try/except to roll back on failure.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx14'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx14',
        'subtopics': ["PyTorch: backward()", "Trainer: step counter increment"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()